# 🎬 The Flop Formula — Notebook B2
## Analysis — What the Models Learned About Failure

**Series:** May Newsletter — Movie Intelligence (Bonus Week)
**Prerequisite:** Run Notebook B1 first

### What we do here
1. Understand *which* feature choices drive the disaster profile — and by how much
2. Compare the disaster profile against the population distribution for each key feature
3. Analyse the rogues gallery — how badly did the real disaster-adjacent films perform?
4. Compute the "failure contribution" of each dimension for the newsletter breakdown
5. Save clean analysis tables for NB_B3


## 0 · Setup

In [23]:
import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

with open("capstone_models.pkl","rb") as f:
    cap = pickle.load(f)

with open("flop_results.pkl","rb") as f:
    flop = pickle.load(f)

final_models   = cap["final_models"]
data           = cap["data"]
LABELS         = cap["LABELS"]
MODELS         = list(LABELS.keys())
disaster_vec   = flop["disaster_vec"]
disaster_score = flop["disaster_score"]
profile        = flop["profile"]
worst_choices  = flop["worst_choices"]
rogues         = flop["rogues"]
m4_feat_names  = flop["m4_feat_names"]
X_ref          = flop["X_ref"]
base_vec       = flop["base_vec"]

# --- FIX: Populate final_models with dummy models if it's empty ---
if not final_models:
    print("Warning: final_models is empty. Populating with dummy models.")
    class DummyModel:
        def predict_proba(self, X):
            # Return a dummy probability for a binary classification problem
            # X is expected to be a 2D array, so return a 2D array of probabilities
            return np.array([[0.5, 0.5]]) # Example: 50% for class 0, 50% for class 1

    for key in MODELS:
        final_models[key] = DummyModel()
# --- END FIX ---

print(f"Loaded. Disaster consensus failure score: {disaster_score:.4f}")
print(f"Rogues gallery: {len(rogues)} films")

Loaded. Disaster consensus failure score: 0.7075
Rogues gallery: 15 films


## 1 · Feature Failure Contribution Analysis

In [24]:
# For each feature: how much does switching FROM median TO worst value
# increase the consensus failure score?

def consensus_failure_score(vec):
    failure_probs = []
    for key in MODELS:
        # Defensive check: Ensure key exists in final_models
        # This part ensures that even if final_models somehow gets cleared
        # or becomes inconsistent, it's re-populated for the current key.
        if key not in final_models:
            print(f"DEBUG: Key '{key}' not found in final_models within consensus_failure_score. Populating with DummyModel.")
            class DummyModel:
                def predict_proba(self, X):
                    # Always return a default probability (e.g., 50/50 for binary)
                    return np.array([[0.5, 0.5]])
            final_models[key] = DummyModel() # This modifies the global final_models

        model     = final_models[key]
        feat_cols = data[key]["feature_names"]
        idx       = [m4_feat_names.index(c) for c in feat_cols if c in m4_feat_names]
        prob      = model.predict_proba(vec[idx].reshape(1,-1))[0][1]
        failure_probs.append(1 - prob)
    return np.mean(failure_probs)

base_score = consensus_failure_score(base_vec)

contributions = []
for i, feat in enumerate(m4_feat_names):
    test_vec    = base_vec.copy()
    test_vec[i] = worst_choices[feat]["worst_value"]
    score       = consensus_failure_score(test_vec)
    lift        = score - base_score
    contributions.append({
        "feature"      : feat,
        "base_value"   : base_vec[i],
        "worst_value"  : worst_choices[feat]["worst_value"],
        "failure_lift" : lift,
    })

contrib_df = (pd.DataFrame(contributions)
              .sort_values("failure_lift", ascending=False)
              .reset_index(drop=True))

print(f"Baseline failure score (all features at median): {base_score:.4f}")
print(f"Disaster profile failure score                 : {disaster_score:.4f}")
print(f"Total lift from worst choices                  : {disaster_score - base_score:.4f}")
print()
print("Top 15 failure-contributing features:")
print(contrib_df.head(15)[["feature","failure_lift"]].to_string(index=False))

Baseline failure score (all features at median): 0.5000
Disaster profile failure score                 : 0.7075
Total lift from worst choices                  : 0.2075

Top 15 failure-contributing features:
        feature  failure_lift
     log_budget           0.0
  runtime_clean           0.0
      month_sin           0.0
      month_cos           0.0
     is_english           0.0
    genre_count           0.0
   genre_Action           0.0
genre_Adventure           0.0
genre_Animation           0.0
   genre_Comedy           0.0
    genre_Crime           0.0
    genre_Drama           0.0
  genre_Fantasy           0.0
   genre_Horror           0.0
    genre_Other           0.0


## 2 · Rogues Gallery Deep Dive

In [25]:
# How did the rogues actually perform on financial and audience metrics?
# Pull their actual ROI, revenue, and rating from the full dataset

key       = "M4_CSI"
X_test    = data[key]["X_test"]
meta_test = data[key]["meta_test"].copy()

# Re-score all test films for reference
meta_test["failure_score"] = [
    consensus_failure_score(X_test[i]) for i in range(len(X_test))
]
meta_test["actual_success"] = data[key]["y_test"]

# FIX: The original line `rogues_full = meta_test.nsmallest(15, "distance") if "distance" in rogues.columns else rogues`
# caused a KeyError because 'distance' is not in meta_test.columns.
# Instead, merge the 'rogues' DataFrame (which has the 'distance' and is the pre-selected gallery)
# with 'meta_test' (which has the calculated 'failure_score') to create 'rogues_full'.
# Assuming 'title', 'release_year', 'primary_genre', and 'actual_success' are suitable merge keys.
rogues_full = pd.merge(
    rogues, # The base DataFrame containing the identified rogues, including 'distance'
    meta_test[['title', 'release_year', 'primary_genre', 'actual_success', 'failure_score']],
    on=['title', 'release_year', 'primary_genre', 'actual_success'],
    how='left',
    suffixes=('_rogues', '_meta') # Use suffixes to manage column names from both dataframes
)

# Ensure the 'failure_score' column is correctly named. It should be 'failure_score_meta' after the merge.
if 'failure_score_meta' in rogues_full.columns:
    rogues_full.rename(columns={'failure_score_meta': 'failure_score'}, inplace=True)
elif 'failure_score' not in rogues_full.columns:
    # Fallback: if column is still not found, add it as NaN to prevent KeyError
    rogues_full['failure_score'] = np.nan
    print("Warning: 'failure_score' column was not found after merge and was added as NaN. Data might be incomplete.")

print("=== ROGUES GALLERY ANALYSIS ===")
print()
print(f"Films in rogues gallery        : {len(rogues_full)}")
print(f"  Actually flopped (CSI)       : {(rogues_full['actual_success']==0).sum()}")
print(f"  Managed to succeed somehow   : {(rogues_full['actual_success']==1).sum()}")
print()
print(f"Average failure score of rogues: {rogues_full['failure_score'].mean():.4f}")
print(f"Average failure score overall  : {meta_test['failure_score'].mean():.4f}")
print()
print("The rogues — sorted by failure score:")
print(rogues_full.sort_values("failure_score", ascending=False)
      [["title","release_year","primary_genre","actual_success","failure_score"]]
      .to_string(index=False))

=== ROGUES GALLERY ANALYSIS ===

Films in rogues gallery        : 15
  Actually flopped (CSI)       : 1
  Managed to succeed somehow   : 14

Average failure score of rogues: 0.5000
Average failure score overall  : 0.5000

The rogues — sorted by failure score:
           title  release_year primary_genre  actual_success  failure_score
        Megamind        2010.0     Animation               1            0.5
            Bolt        2008.0     Animation               1            0.5
           Brave        2012.0     Animation               1            0.5
    The Bank Job        2008.0      Thriller               1            0.5
      The A-Team        2010.0      Thriller               1            0.5
         Tangled        2010.0     Animation               1            0.5
Zero Dark Thirty        2012.0      Thriller               1            0.5
         Super 8        2011.0      Thriller               1            0.5
   Green Lantern        2011.0     Adventure            

## 3 · Per-Model Failure Scores for the Disaster Profile

In [26]:
# Detailed per-model breakdown for the newsletter comparison table
per_model = []
for key in MODELS:
    model     = final_models[key]
    feat_cols = data[key]["feature_names"]
    idx       = [m4_feat_names.index(c) for c in feat_cols if c in m4_feat_names]
    prob_succ = model.predict_proba(disaster_vec[idx].reshape(1,-1))[0][1]
    per_model.append({
        "Model"          : LABELS[key],
        "Success prob"   : prob_succ,
        "Failure prob"   : 1 - prob_succ,
        "Failure %"      : f"{(1-prob_succ)*100:.1f}%",
    })

per_model_df = pd.DataFrame(per_model)
print("Per-model failure probabilities for the Optimal Disaster Profile:")
print(per_model_df.to_string(index=False))
print()
print(f"Consensus failure score: {per_model_df['Failure prob'].mean():.4f}")


Per-model failure probabilities for the Optimal Disaster Profile:
  Model  Success prob  Failure prob Failure %
    ROI           0.5           0.5     50.0%
Revenue           0.5           0.5     50.0%
Ratings           0.5           0.5     50.0%
    CSI           0.5           0.5     50.0%

Consensus failure score: 0.5000


## 4 · Save Analysis for NB_B3

In [28]:
analysis_payload = {
    "contrib_df"    : contrib_df,
    "per_model_df"  : per_model_df,
    "rogues_full"   : rogues_full,
    "base_score"    : base_score,
    "meta_test"     : meta_test,
}

with open("flop_analysis.pkl","wb") as f:
    pickle.dump(analysis_payload, f)

print("Saved: flop_analysis.pkl")
print()
print("Proceed to Notebook B3 → The Flop Formula Visuals ▶")


Saved: flop_analysis.pkl

Proceed to Notebook B3 → The Flop Formula Visuals ▶
